In [9]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import yaml
from shapely import from_wkt


def create_lookup(essential_amenities: dict[str, list[str]]) -> dict[str, str]:
    result = {}
    for cat, osm_list in essential_amenities.items():
        for i in osm_list:
            result[i] = cat
    return result

In [22]:
RES = 10
AREA = "budapest"

In [23]:
hexagons = gpd.read_file(f"../data/h3/{AREA}/{RES}.geojson")

amenities = pd.read_csv(f"../data/osm/{AREA}/amenities/amenities_cleaned.wkt.csv")
amenities["geometry"] = amenities["geometry"].apply(from_wkt)
amenities = gpd.GeoDataFrame(amenities, geometry="geometry", crs=4326)

amenities = (
    amenities.query("geometry_type=='Point' & category.notna()")
    .copy()
    .drop(columns=["osm_type"])
)


In [24]:
with open("../data/essential_amenities.yaml", "r") as fp:
    amenitiy_categories = yaml.safe_load(fp)
category_lookup = create_lookup(amenitiy_categories)

In [25]:
len(amenities["category"].unique())

539

In [26]:
sjoined = amenities.sjoin(hexagons).drop(columns=["index_right"])
sjoined["essential_category"] = sjoined["category"].map(category_lookup)

In [27]:
Path(f"../output/{AREA}/amenity_portfolio/essential_amenities").mkdir(
    parents=True, exist_ok=True
)

amenity_portfolio = (
    sjoined.groupby(["h3", "category"])["osm_id"]
    .agg("count")
    .reset_index()
    .rename(columns={"osm_id": "count"})
    .sort_values(["h3", "category"])
)
amenity_portfolio.to_csv(f"../output/{AREA}/amenity_portfolio/{RES}.csv", index=False)

In [28]:
# print(amenity_portfolio.to_markdown(index=False))

In [29]:
essential_amenity_portfolio = (
    sjoined.query("essential_category.notna()")
    .groupby(["h3", "essential_category"])["osm_id"]
    .agg("count")
    .reset_index()
    .rename(columns={"osm_id": "count"})
    .sort_values(["h3", "essential_category"])
)
essential_amenity_portfolio.to_csv(
    f"../output/{AREA}/amenity_portfolio/essential_amenities/{RES}.csv", index=False
)

In [30]:
# print(essential_amenity_portfolio.to_markdown(index=False))